# DEC5 shared-rig reconstruction control: comparison integrity
Unit: one temporal reconstruction (000973), three held cameras F/J/L. Face PSNR/SSIM/LPIPS are computed only for F with the same GT-only polygon. The 310 train EXRs span three fit times and two held times. This is not a temporal campaign pass or a full-frame metric report.
The selected regularized focal calibration is rejected: native crops retain skin/tube/hair defects and show absolute projection displacement. Improvements to held epipolar residuals alone do not establish physical calibration accuracy. The exact standalone audit is `/mnt/data/lookcloser_dec5_5a3_surface_repair/tools/audit_shared_rig_control.py`.

In [1]:
from pathlib import Path
import hashlib, json, math
root = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair/calibration_control')
audit = json.loads((root/'shared_rig_findings.json').read_text())
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
for path, expected in audit['retained_hashes'].items():
    assert digest(path) == expected, path
assert audit['source_EXRs_verified'] == 310
assert not set(audit['fit_times']) & set(audit['held_times'])
assert not audit['calibration_accepted'] and not audit['reconstruction_accepted']
print('Verified hashes:', len(audit['retained_hashes']))
print(audit['original_runtime_replay'])


Verified hashes: 485
Three F/J/L images rerendered with candidate runtime; all SHA-256 equal the historical control.


In [2]:
old = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair/diagnostics/000973/color_refit_exact_three_views')
new = root/'reconstruct_000973'
metrics = [json.loads((p/'face_metrics/metrics.json').read_text()) for p in [old,new]]
for key in ['ground_truth_sha256','face_polygons_sha256','face_bbox_xyxy','protocol','metric_domain']:
    assert metrics[0][key] == metrics[1][key], key
print('Variant | Face PSNR | Face SSIM | Face LPIPS')
for row in audit['face_metrics']:
    values = [row[k] for k in ['face_psnr','face_ssim','face_lpips']]
    assert all(math.isfinite(v) for v in values)
    print(row['variant'], *[round(v,6) for v in values], sep=' | ')
assert audit['depth_maps'] == 62 and audit['depth_shape'] == [1080,1920]
assert audit['mesh_vertices'] > 0 and audit['mesh_triangles'] > 0 and audit['mesh_components'] == 1
review = json.loads((new/'visual_review.json').read_text())
assert review['status'] == 'fail' and all(v['status'] == 'fail' for v in review['views'])
assert len(review['inspected_crops']) == 8
print('Native review: 0 pass / 3 fail; eight crops viewed. No temporal promotion.')


Variant | Face PSNR | Face SSIM | Face LPIPS
original_rig_matched_renderer | 28.724285 | 0.889098 | 0.047989
shared_focal_regularized | 16.915396 | 0.655352 | 0.417596
Native review: 0 pass / 3 fail; eight crops viewed. No temporal promotion.


## Interpretation and limits
Comparison integrity passes, candidate quality fails (high confidence from actual crops). The same GT/polygon excludes ROI drift as the explanation for this particular metric regression. Original-rig F/J/L rerenders are byte-identical under the current runtime, excluding optional disabled renderer additions as a confound. A lower shared-rig epipolar error is not enough to maintain geometry in the coordinate system of unchanged held cameras. Why the candidate deforms the recovered projection still requires an absolute, train-camera-held-out calibration/gauge check. Do not fit or warp F/J/L to their GT to conceal this failure. Original source calibration and published campaign are unchanged.